# Chương 13. Phân tích khám phá dữ liệu

**Câu hỏi mở đầu:** Khi chưa biết rõ câu trả lời, ta khám phá một tập dữ liệu có hệ thống như thế nào?

Notebook này là tài nguyên đồng hành của chương. Mỗi phần đều đi theo nhịp **câu hỏi → dữ liệu → mã → kết quả → diễn giải → kiểm tra bằng chứng**.

## Mục tiêu

- Tái hiện các ví dụ cốt lõi của chương bằng mã có thể chạy lại.
- Kiểm tra giả định trước khi diễn giải output.
- Kết thúc bằng ít nhất một câu hỏi về điều mà kết quả **chưa** cho biết.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("error", category=FutureWarning)
warnings.filterwarnings("error", category=DeprecationWarning)
ROOT = Path.cwd()
DATA = ROOT / "data"
print("Working root:", ROOT)


In [ ]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 20)


In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 100


In [ ]:
tx=pd.read_csv(DATA / "metromart" / "metromart_transactions.csv")
prod=pd.read_csv(DATA / "metromart" / "metromart_products.csv")
work=tx.merge(prod[["product_id","category"]],on="product_id",validate="many_to_one")
valid=work["unit_price"].notna() & work["discount_pct"].between(0,1)
work=work.loc[valid].copy()

## 1. Vòng lặp EDA: kiểm tra → tóm tắt → trực quan → so sánh → câu hỏi mới

In [ ]:
print(work.shape)
print(work[["unit_price","quantity"]].describe())
print(work["category"].value_counts().head())

## 2. Mẫu hình toàn bộ dữ liệu

In [ ]:
r_all=work["unit_price"].corr(work["quantity"])
print("r_all =",r_all)

## 3. Kiểm tra theo biến thứ ba

In [ ]:
r_by_category=(work.groupby("category")
    .apply(lambda g: g["unit_price"].corr(g["quantity"]), include_groups=False)
    .rename("r"))
display(r_by_category.sort_values())

Nếu tương quan toàn bộ và tương quan trong nhóm khác nhau, nhận định ban đầu phải được sửa. EDA dùng mẫu hình để tạo câu hỏi mới, không tự động xác nhận nguyên nhân.

## 4. Một hình kiểm tra

In [ ]:
sample=work.sample(900,random_state=11)
fig,ax=plt.subplots(figsize=(6,4))
for name,g in sample.groupby("category"):
    ax.scatter(g["unit_price"],g["quantity"],s=10,alpha=.35,label=name)
ax.set_xlabel("Đơn giá")
ax.set_ylabel("Số lượng")
ax.set_title("Đơn giá và lượng bán theo nhóm sản phẩm")
ax.legend(fontsize=7,ncol=2)
plt.show()

### Bản ghi nhớ EDA

Mỗi phát hiện cần bốn dòng: **Phát hiện — Bằng chứng — Giới hạn — Câu hỏi tiếp theo**.

## Thực hành

Viết 3 phát hiện, 2 bất thường, 2 câu hỏi mới và 1 giới hạn từ các output trên.

---
### Bạn đã sẵn sàng sang chương tiếp theo nếu có thể…

- giải thích output bằng lời;
- chỉ ra ít nhất một giả định;
- nói được kết quả chưa cho phép kết luận điều gì.

**Exit check:** Nếu mã chạy không lỗi nhưng câu trả lời trái với ý nghĩa của dữ liệu, bạn sẽ kiểm tra điều gì trước?